In [ ]:
from keras.layers import Lambda, Input, Dense
from keras.models import Model
from keras.losses import mse, binary_crossentropy
from keras import backend as K
from sklearn.decomposition import PCA
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import random
import tensorflow as tf

cmap='tab10'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!ls /content/drive/MyDrive/dataset

english-all.csv				MandarinBeijing_WS_20230907_shaped.csv
english_american.csv			MandarinBeijing_WS_20230907_shaped_format.csv
EnglishAmerican_WS_20240411_shaped.csv	mandarin.csv
english.csv				VAEloss中国語new.csv
JMCDI_SPSE_1_ageM.csv			VAEloss英語_20240411bkup.csv
JMCDI_SPSE_2_ageM.csv			VAEloss英語_20240411ver0.0.csv
mandarin_beijing.csv			VAEloss英語5.csv


**Supplementary Materials execution note (added in 2026)**

This notebook documents the English 3D-VAE seed search: 50 trials followed by 950 further trials. It ends with the export of `seed`, `loss`, and `val_loss` to `VAEloss_English_20240411ver0.1.csv`. Later cells retained from an earlier experiment were omitted from this Supplementary Materials version because they do not document the final model. The accompanying final-model notebook reads this CSV, ranks the trials by validation loss, and selects seed 26172.

**Editorial note (2026).** This Supplementary Materials copy was prepared from the archived Colab notebook. Explanatory comments were translated into English, and the loss CSV filename was standardized to `VAEloss_English_20240411ver0.1.csv`. An obsolete triple-quoted code block for creating vocabulary/category lists and its associated saved output were removed from the data-formatting cell. The notebook has not been re-executed after these editorial changes. Execution counts are left unset (`null`); any retained outputs are historical records, not results of a new run. The original notebook is preserved separately.

**Data preprocessing**

In [ ]:
# Define this function for reproducibility before calling fit
# Setting the seed alone did not reproduce the same results, so this additional setup was adopted from an online source

def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

    session_conf = tf.compat.v1.ConfigProto(
        intra_op_parallelism_threads=1,
        inter_op_parallelism_threads=1
    )
    sess = tf.compat.v1.Session(graph=tf.compat.v1.get_default_graph(), config=session_conf)
    tf.compat.v1.keras.backend.set_session(sess)

In [ ]:
"""

# Support for wide-format data
df = pd.read_csv('/content/drive/MyDrive/dataset/EnglishAmerican_WS_20230907_reconstructed_wide.csv', index_col=1)# English
df=df.fillna(0)# Replace missing values with 0
age=df.age.values
df=df.replace(['produces'], 1)# Convert responses to binary (0/1) data
df=df.drop(['downloaded',"child_id","age"],axis=1)
dim=len(df.values[0])

"""

<ipython-input-3-b22934011394>:2: DtypeWarning: Columns (175,181,187,262,269,280,283,313,318,329,331,333,346,375,411,419,421,428,434,439,441,452,454,460,463,469,475,481,485,490,493,494,504,539,544,552,563,570,571,577,578,579,581,582,583,592,593,594,596,597,598,599,601,602,606,609,611,615,620,621,627,628,629,632,634,636,637,639,643,645,646,647,649,653,658,659,661,664,666,667,668,672,674,675,676,677,678,680,681,682,683) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('/content/drive/MyDrive/dataset/EnglishAmerican_WS_20230907_reconstructed_wide.csv', index_col=1)#英語


In [ ]:
## Code to format data downloaded from Wordbank ##
## Commented out because this is not currently needed ##



#df = pd.read_csv('/content/drive/MyDrive/dataset/mandarin_beijing.csv', index_col=0)# Mandarin Chinese
df = pd.read_csv('/content/drive/MyDrive/dataset/EnglishAmerican_WS_20240411_shaped.csv', index_col=1)# English
df=df.fillna(0)# Replace missing values with 0
df=df[df['item_kind'] =='word']# Keep only word items
df=df.replace('produces', 1)# Convert responses to binary (0/1)
index=list(dict.fromkeys(df.index))# Obtain index values
index=[int(item) for item in index]
n=len(index)# Number of children
dim=len(df.loc[index[0]])# Number of items
age=df.age.values.reshape(n,dim).T[0]

# Create headers and an index
import string
import collections
cate=list(dict.fromkeys(df.category[index[0]]))# List of categories
name=[chr(i) for i in range(65, 65+len(cate))]# Generate labels A, B, C, etc.

header=[]
for na,i in enumerate(cate):
  for j in range(collections.Counter(df.category[index[0]])[i]):
    header.append(name[na]+str(j+1))



In [ ]:
df=pd.DataFrame(data=df.value.values.reshape(n,dim),index=index,columns=header)

In [ ]:
df

,A1,A2,A3,A4,A5,A6,A7,A8,A9,A10,...,U18,U19,U20,U21,V1,V2,V3,V4,V5,V6
396587,1,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,0,0,1,1,1
396588,1,1,1,1,1,1,1,1,1,1,...,1,0,0,0,1,0,1,0,0,0
396589,1,1,1,1,1,1,1,1,1,1,...,1,0,1,0,0,0,0,0,0,0
396590,1,1,0,1,0,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
396591,1,1,0,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
406721,1,1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
406724,1,1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
406727,1,1,1,1,1,1,0,1,1,0,...,1,0,1,0,1,1,1,0,1,1
406729,0,1,0,1,0,1,1,0,1,1,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
# Examine various seeds. The list is kept at 20 for lighter computation, but 100 or 200 are possible; English takes longer because of its larger sample size
seed_list=[random.randint(0, 100000) for i in range(50)]

In [ ]:
# Save results incrementally because long runs may be interrupted
# Create the initial CSV file

#chunk_size = len(seed_list) // 10  # When splitting into 10 parts
chunk_size = len(seed_list)   # Do not split the list
# Split the list into 10 parts and create a nested list
divided_list = [seed_list[i:i + chunk_size] for i in range(0, len(seed_list), chunk_size)]
dim=680

for j in range(1):
  loss_list=[]
  val_loss_list=[]
  for i in divided_list[j]:
    seed_everything(i)
    # Split the test set without stratifying by age in months
    # Following the MNIST example, divide the data into seven parts
    test=df.sample(frac=1/7)
    tesindex=test.index
    train=df.drop(test.index)


    # Set model parameters
    input_shape = (dim, )
    intermediate_dim = 512
    batch_size = 128
    latent_dim = 3

    # Build encoder: a fully connected layer followed by z_mean and z_log_var layers
    inputs = Input(shape=input_shape, name='encoder_input')
    x = Dense(intermediate_dim, activation='relu')(inputs)
    z_mean = Dense(latent_dim, name='z_mean')(x)
    z_log_var = Dense(latent_dim, name='z_log_var')(x)

    # Sample the latent variable z
    def sampling(args):
        z_mean, z_log_var = args
        batch = K.shape(z_mean)[0]
        dim = K.int_shape(z_mean)[1]
        epsilon = K.random_normal(shape=(batch, dim))
        return z_mean + K.exp(0.5 * z_log_var) * epsilon
    z = Lambda(sampling, output_shape=(latent_dim,), name='z')([z_mean, z_log_var])


    encoder = Model(inputs, [z_mean, z_log_var, z], name='encoder')
    #encoder.summary()


    # Build decoder: a fully connected layer and a sigmoid-activated output layer
    latent_inputs = Input(shape=(latent_dim,), name='z_sampling')
    x = Dense(intermediate_dim, activation='relu')(latent_inputs)
    outputs = Dense(dim, activation='sigmoid')(x)

    decoder = Model(latent_inputs, outputs, name='decoder')
    #decoder.summary()

    outputs = decoder(encoder(inputs)[2])
    vae = Model(inputs, outputs)

    # Define the loss function
    # VAE loss = mse_loss or xent_loss + kl_loss
    reconstruction_loss = binary_crossentropy(inputs,outputs)*dim

    kl_loss = 1 + z_log_var - K.square(z_mean) - K.exp(z_log_var)
    kl_loss = -0.5*K.sum(kl_loss, axis=-1)

    vae_loss = K.mean(reconstruction_loss + kl_loss)
    vae.add_loss(vae_loss)
    # Use Adam as the optimizer
    vae.compile(optimizer='adam')
    #vae.summary()

    # Train the model
    result=vae.fit(train,
              epochs=30,
              batch_size=batch_size,
              validation_data=(test, None))

    loss_list.append(result.history["loss"][-1])
    val_loss_list.append(result.history["val_loss"][-1])

  loss_data=pd.DataFrame({"seed":divided_list[j],"loss":loss_list,"val_loss":val_loss_list})
  loss_data.to_csv('/content/drive/MyDrive/dataset/VAEloss_English_20240411ver0.1.csv',index=None)

Epoch 1/30
61/61 [==============================] - 4s 8ms/step - loss: 315.8568 - val_loss: 216.4196
Epoch 2/30
61/61 [==============================] - 0s 5ms/step - loss: 214.9771 - val_loss: 205.8944
Epoch 3/30
61/61 [==============================] - 0s 5ms/step - loss: 208.9983 - val_loss: 202.2599
Epoch 4/30
61/61 [==============================] - 0s 5ms/step - loss: 205.5229 - val_loss: 200.1068
Epoch 5/30
61/61 [==============================] - 0s 5ms/step - loss: 203.1600 - val_loss: 198.4420
Epoch 6/30
61/61 [==============================] - 0s 5ms/step - loss: 202.2663 - val_loss: 197.7249
Epoch 7/30
61/61 [==============================] - 0s 5ms/step - loss: 201.8111 - val_loss: 196.9555
Epoch 8/30
61/61 [==============================] - 0s 5ms/step - loss: 201.2219 - val_loss: 196.5233
Epoch 9/30
61/61 [==============================] - 0s 5ms/step - loss: 200.7702 - val_loss: 195.9833
Epoch 10/30
61/61 [==============================] - 0s 5ms/step - loss: 200.3531 

In [ ]:
seed_list=[random.randint(0, 100000) for i in range(950)]

In [ ]:
# Repeatedly load the existing CSV, append results, and save
# There may be a better way to implement this

chunk_size = len(seed_list) // 10  # When splitting into 10 parts
# Split the list into 10 parts and create a nested list
divided_list = [seed_list[i:i + chunk_size] for i in range(0, len(seed_list), chunk_size)]
dim=680

for j in range(10):
  loss_list=[]
  val_loss_list=[]
  for i in divided_list[j]:
    seed_everything(i)
    # Split the test set without stratifying by age in months
    # Following the MNIST example, divide the data into seven parts
    test=df.sample(frac=1/7)
    tesindex=test.index
    train=df.drop(test.index)


    # Set model parameters
    input_shape = (dim, )
    intermediate_dim = 512
    batch_size = 128
    latent_dim = 3

    # Build encoder: a fully connected layer followed by z_mean and z_log_var layers
    inputs = Input(shape=input_shape, name='encoder_input')
    x = Dense(intermediate_dim, activation='relu')(inputs)
    z_mean = Dense(latent_dim, name='z_mean')(x)
    z_log_var = Dense(latent_dim, name='z_log_var')(x)

    # Sample the latent variable z
    def sampling(args):
        z_mean, z_log_var = args
        batch = K.shape(z_mean)[0]
        dim = K.int_shape(z_mean)[1]
        epsilon = K.random_normal(shape=(batch, dim))
        return z_mean + K.exp(0.5 * z_log_var) * epsilon
    z = Lambda(sampling, output_shape=(latent_dim,), name='z')([z_mean, z_log_var])


    encoder = Model(inputs, [z_mean, z_log_var, z], name='encoder')
    #encoder.summary()


    # Build decoder: a fully connected layer and a sigmoid-activated output layer
    latent_inputs = Input(shape=(latent_dim,), name='z_sampling')
    x = Dense(intermediate_dim, activation='relu')(latent_inputs)
    outputs = Dense(dim, activation='sigmoid')(x)

    decoder = Model(latent_inputs, outputs, name='decoder')
    #decoder.summary()

    outputs = decoder(encoder(inputs)[2])
    vae = Model(inputs, outputs)

    # Define the loss function
    # VAE loss = mse_loss or xent_loss + kl_loss
    reconstruction_loss = binary_crossentropy(inputs,outputs)*dim

    kl_loss = 1 + z_log_var - K.square(z_mean) - K.exp(z_log_var)
    kl_loss = -0.5*K.sum(kl_loss, axis=-1)

    vae_loss = K.mean(reconstruction_loss + kl_loss)
    vae.add_loss(vae_loss)
    # Use Adam as the optimizer
    vae.compile(optimizer='adam')
    #vae.summary()

    # Train the model
    result=vae.fit(train,
              epochs=30,
              batch_size=batch_size,
              validation_data=(test, None))

    loss_list.append(result.history["loss"][-1])
    val_loss_list.append(result.history["val_loss"][-1])


  # Append new results to the existing CSV containing seed, loss, and val_loss
  tmp=pd.read_csv('/content/drive/MyDrive/dataset/VAEloss_English_20240411ver0.1.csv')
  loss_data=pd.DataFrame({"seed":divided_list[j],"loss":loss_list,"val_loss":val_loss_list})
  tmp=pd.concat([tmp,loss_data])
  tmp.to_csv('/content/drive/MyDrive/dataset/VAEloss_English_20240411ver0.1.csv',index=None)

ストリーミング出力は最後の 5000 行に切り捨てられました。
61/61 [==============================] - 0s 5ms/step - loss: 197.7114 - val_loss: 195.2932
Epoch 22/30
61/61 [==============================] - 0s 5ms/step - loss: 197.6775 - val_loss: 195.3063
Epoch 23/30
61/61 [==============================] - 0s 5ms/step - loss: 197.5354 - val_loss: 196.2083
Epoch 24/30
61/61 [==============================] - 0s 5ms/step - loss: 197.6477 - val_loss: 195.3577
Epoch 25/30
61/61 [==============================] - 0s 5ms/step - loss: 197.2892 - val_loss: 194.9468
Epoch 26/30
61/61 [==============================] - 0s 5ms/step - loss: 197.1424 - val_loss: 195.0560
Epoch 27/30
61/61 [==============================] - 0s 5ms/step - loss: 197.0620 - val_loss: 194.8643
Epoch 28/30
61/61 [==============================] - 0s 5ms/step - loss: 197.0404 - val_loss: 195.0461
Epoch 29/30
61/61 [==============================] - 0s 5ms/step - loss: 196.8194 - val_loss: 194.8148
Epoch 30/30
61/61 [==============================] - 